# Highway 97 Road Segment Congestion

This notebook downloads recent GTFS-Realtime vehicle and trip updates, runs the reusable congestion engine in `forecasting`, and visualizes direction-specific segment congestion. Segments follow each Route 97 trip's shape and ordered stops. Vehicle updates must match their own trip, direction, stop ID, and stop sequence; opposite directions and different shapes are never pooled.

## Requirements

Install the forecasting package and notebook dependencies from the repository (`cd forecasting && pip install -r requirements.txt`). Create the ignored `.env.road_segment_congestion` file by sourcing `./setup_road_segment_congestion_env.sh` from the repository root. This notebook loads the five `TRANSIT_DB_*` variables from that file automatically. Credentials are intentionally not stored in the notebook. The realtime pull below uses the most recent 24 hours. Internet access is also required to fetch current static GTFS for the same Kelowna operator.

## Packages

In [ ]:
import os
from pathlib import Path
import shlex
import sys

import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from forecasting import (
    download_backtest_data,
    get_latest_stop_delays,
    run_congestion_engine,
)
from forecasting.static_gtfs import download_static_gtfs

## Constants

In [ ]:


workspace_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
repo_root = next(
    (
        candidate
        for candidate in workspace_candidates
        if (candidate / 'data' / 'static_data').is_dir()
        and (candidate / 'forecasting' / 'forecasting').is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository data and forecasting package.')

env_file = repo_root / '.env.road_segment_congestion'
if not env_file.is_file():
    raise FileNotFoundError(
        f'{env_file} is missing. From the repository root, run '
        '`source ./setup_road_segment_congestion_env.sh` first.'
    )

required_env_vars = {
    'TRANSIT_DB_HOST',
    'TRANSIT_DB_USER',
    'TRANSIT_DB_PASSWORD',
    'TRANSIT_DB_NAME',
    'TRANSIT_DB_PORT',
}
for line_number, raw_line in enumerate(env_file.read_text().splitlines(), start=1):
    line = raw_line.strip()
    if not line or line.startswith('#'):
        continue
    if line.startswith('export '):
        line = line.removeprefix('export ').strip()
    variable, separator, encoded_value = line.partition('=')
    if not separator or variable not in required_env_vars:
        raise ValueError(f'Invalid environment setting on line {line_number} of {env_file}.')
    parsed_value = shlex.split(encoded_value, posix=True)
    if len(parsed_value) != 1:
        raise ValueError(f'Invalid value on line {line_number} of {env_file}.')
    os.environ[variable] = parsed_value[0]

missing_env_vars = sorted(required_env_vars - os.environ.keys())
if missing_env_vars:
    raise RuntimeError('Missing database settings: ' + ', '.join(missing_env_vars))



static_data_dir = repo_root / 'data' / 'static_data'

visuals_dir = repo_root / 'notebooks' / 'visuals'
visuals_dir.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo_root}')

## Download backtest data

Both update tables are downloaded newest-first. The congestion calculations use vehicle updates; trip updates are also returned for delay analyses.

The following static-data cell fetches current BC Transit GTFS for operator 47 into the ignored `data/static_gtfs_cache/` directory. Every archive gets its own versioned snapshot, so the historical `data/static_data/` files remain unchanged. The publisher is contacted on every run; network failures do not silently fall back to an old snapshot. For September backtesting, explicitly load the saved September realtime data together with `data/static_data/` instead of mixing historical GTFS with live updates.

In [ ]:
df_vehicle_updates, df_trip_updates = download_backtest_data(lookback_hours=24*2)
print(f'Vehicle updates: {len(df_vehicle_updates):,}')
print(f'Trip updates: {len(df_trip_updates):,}')
print(
    'Vehicle update window: '
    f'{df_vehicle_updates["feed_timestamp"].min()} through '
    f'{df_vehicle_updates["feed_timestamp"].max()}'
)

In [ ]:
static_data_dir = download_static_gtfs(repo_root / 'data' / 'static_gtfs_cache')
print(f'Static GTFS snapshot: {static_data_dir}')
feed_info_path = static_data_dir / 'feed_info.txt'
if feed_info_path.is_file():
    print(pd.read_csv(feed_info_path, dtype='string').to_string(index=False))

df_stops = pd.read_csv(static_data_dir / 'stops.txt')
df_stop_times = pd.read_csv(static_data_dir / 'stop_times.txt')
df_trips = pd.read_csv(static_data_dir / 'trips.txt')
df_shapes = pd.read_csv(static_data_dir / 'shapes.txt')

print(
    f'Loaded {len(df_stops):,} stops, {len(df_stop_times):,} stop times, '
    f'{len(df_trips):,} trips, and {len(df_shapes):,} shape points.'
)

## Run the congestion engine

The package constructs directed stop-to-stop segments for every selected Route 97 trip and shape. Updates must match `(trip_id, trip_direction_id, stop_id, current_stop_sequence)` against that trip's GTFS stop times. The same directed stop pair on the same shape shares a segment; opposite directions, express stop pairs, and different shapes remain separate.

Set `trip_direction_id = None` below to process both directions separately, or choose `0` or `1`. Segment IDs are unique within each result, not permanent GTFS identifiers. The reference trip is retained for inspection; it is no longer the only source of segments. The route geometry includes all selected shapes.

Unmatched updates are skipped with an explicit warning, rather than being assigned by a sequence number from another trip. Use a static GTFS snapshot aligned with the realtime data period. Updates naming a trip's origin stop have no incoming segment and are excluded.

Each directed segment and shape has its own median historical speed baseline. Mean speed is aggregated into five-minute UTC bins, and the congestion index is `1 - observed_speed / baseline_speed`, clipped to 0 (free flow) through 1 (stopped). Baselines at or below 0.5 are excluded because near-stationary layover speeds make the ratio unstable.

In [ ]:
route_id = '97-KEL'
time_bin_size = '5min'
trip_direction_id = None

In [ ]:

congestion_result = run_congestion_engine(
    df_vehicle_updates,
    df_stops,
    df_stop_times,
    df_trips,
    df_shapes,
    route_id=route_id,
    time_bin_size=time_bin_size,
    trip_direction_id=trip_direction_id,
)

segments = congestion_result.segments
route_geometry = congestion_result.route_geometry
vehicle_probes = congestion_result.vehicle_probes
df_congestion = congestion_result.congestion

print(f'Reference trip: {congestion_result.representative_trip_id}')
print(f'Route segments: {len(segments)}')
print(f'Matched vehicle probes: {len(vehicle_probes):,}')
print(f'Congestion estimates: {len(df_congestion):,}')
print('Segments per direction:')
print(segments.groupby('trip_direction_id').size())
print('Matched probes per direction:')
print(vehicle_probes.groupby('trip_direction_id').size())
if df_congestion.empty:
    if vehicle_probes.empty:
        raise ValueError(
            'No valid incoming-segment probes matched the selected GTFS trips. '
            'Check the realtime window, selected direction, and static GTFS snapshot. '
            'Matching requires trip_id, trip_direction_id, stop_id, and stop sequence. '
            f'Loaded static GTFS: {static_data_dir}'
        )
    raise ValueError(
        f'{len(vehicle_probes):,} probes matched, but no segment baseline speed '
        'exceeded 0.5. Select a data window containing moving buses.'
    )
segments[
    ['segment_id', 'trip_direction_id', 'shape_id',
     'from_stop_id', 'to_stop_id', 'from_stop_name', 'to_stop_name']
]

## Latest segment congestion

Each directed line is colored using its latest congestion estimate. Gray indicates no reading in the selected window. Use the direction dropdown to show both directions or isolate one; hover labels include direction and shape IDs.

In [ ]:
latest_congestion = (
    df_congestion.sort_values('time_bin')
    .groupby('segment_id')
    .tail(1)
    .set_index('segment_id')['congestion_index']
)
colorscale = pc.get_colorscale('RdYlGn_r')
direction_buttons = [
    dict(
        label='All selected directions',
        method='restyle',
        args=[{'visible': [True] * len(segments)}],
    )
]
for direction_id in sorted(segments['trip_direction_id'].unique()):
    direction_buttons.append(
        dict(
            label=f'Direction {direction_id}',
            method='restyle',
            args=[{'visible': segments['trip_direction_id'].eq(direction_id).tolist()}],
        )
    )

def congestion_to_color(value):
    if pd.isna(value):
        return 'gray'
    return pc.sample_colorscale(colorscale, [float(value)])[0]

fig = go.Figure()
for _, segment in segments.iterrows():
    segment_id = segment['segment_id']
    from_stop_name = segment['from_stop_name']
    to_stop_name = segment['to_stop_name']
    congestion = latest_congestion.get(segment_id, float('nan'))
    longitude, latitude = segment['geometry'].xy
    hover_text = (
        f'Direction {segment["trip_direction_id"]} | segment {segment_id}: '
        f'{from_stop_name} -> {to_stop_name}'
        f'<br>Shape {segment["shape_id"]}'
        + (
            f'<br>Congestion index: {congestion:.2f}'
            if pd.notna(congestion)
            else '<br>No recent data'
        )
    )
    fig.add_trace(
        go.Scattermap(
            lon=list(longitude),
            lat=list(latitude),
            mode='lines',
            line=dict(width=5, color=congestion_to_color(congestion)),
            name=f'Direction {segment["trip_direction_id"]}: {from_stop_name} -> {to_stop_name}',
            hovertext=hover_text,
            hoverinfo='text',
        )
    )

fig.update_layout(
    title=f'Route {route_id} Directed Segment Congestion (latest reading)',
    updatemenus=[dict(type='dropdown', buttons=direction_buttons, x=0.02, y=1.05)],
    map_style='open-street-map',
    map=dict(
        center=dict(lat=route_geometry.centroid.y, lon=route_geometry.centroid.x),
        zoom=11,
    ),
    showlegend=False,
    width=1500,
    height=700,
)
fig.write_html(visuals_dir / 'route_97_segment_congestion_engine.html')
fig.show()

## Spatial congestion animations

The following maps show the latest day in the downloaded data. The direction dropdown isolates a traffic flow without combining its readings with the other direction. Values are forward-filled within each directed segment and shape, and the smoothed version uses a centered 10-bin rolling average for visualization; it is not a real-time causal estimate.

In [ ]:
animation_window = pd.Timedelta(days=1)
max_time_bin = df_congestion['time_bin'].max()
df_animation = df_congestion.loc[
    df_congestion['time_bin'] >= max_time_bin - animation_window
].copy()

raw_congestion_grid = (
    df_animation.pivot(
        index='time_bin',
        columns='segment_id',
        values='congestion_index',
    )
    .sort_index()
    .reindex(columns=segments['segment_id'])
    .ffill()
)
smoothed_congestion_grid = raw_congestion_grid.rolling(
    window=10,
    min_periods=1,
    center=True,
).mean()

def make_congestion_animation(congestion_grid, title, value_label):
    """Animate the selected directed road segments.

    Parameters
    ----------
    congestion_grid : pandas.DataFrame
        Time bins by unique directed segment ID.
    title : str
        Figure title.
    value_label : str
        Congestion label used in hover text.

    Returns
    -------
    plotly.graph_objects.Figure
        Animation with independent direction controls.
    """
    if congestion_grid.empty:
        raise ValueError('No congestion data is available to animate.')

    frames = []
    for time_bin, row_values in congestion_grid.iterrows():
        frame_data = []
        for _, segment in segments.iterrows():
            segment_id = segment['segment_id']
            from_stop_name = segment['from_stop_name']
            to_stop_name = segment['to_stop_name']
            congestion = row_values[segment_id]
            longitude, latitude = segment['geometry'].xy
            hover_text = (
                f'Direction {segment["trip_direction_id"]} | segment {segment_id}: '
                f'{from_stop_name} -> {to_stop_name}'
                f'<br>Shape {segment["shape_id"]}'
                + (
                    f'<br>{value_label}: {congestion:.2f}'
                    if pd.notna(congestion)
                    else '<br>No data yet'
                )
            )
            frame_data.append(
                go.Scattermap(
                    lon=list(longitude),
                    lat=list(latitude),
                    mode='lines',
                    line=dict(
                        width=5,
                        color=congestion_to_color(congestion),
                    ),
                    name=f'Direction {segment["trip_direction_id"]}: {from_stop_name} -> {to_stop_name}',
                    hovertext=hover_text,
                    hoverinfo='text',
                )
            )
        frames.append(go.Frame(name=str(time_bin), data=frame_data))

    figure = go.Figure(data=frames[0].data, frames=frames)
    figure.update_layout(
        title=title,
        map_style='open-street-map',
        map=dict(
            center=dict(
                lat=route_geometry.centroid.y,
                lon=route_geometry.centroid.x,
            ),
            zoom=11,
        ),
        showlegend=False,
        width=1500,
        height=700,
        updatemenus=[
            {
                'type': 'buttons',
                'showactive': False,
                'x': 0.0,
                'y': 1.05,
                'buttons': [
                    {
                        'label': 'Play',
                        'method': 'animate',
                        'args': [
                            None,
                            {
                                'frame': {'duration': 250, 'redraw': True},
                                'fromcurrent': True,
                                'transition': {'duration': 0},
                            },
                        ],
                    },
                    {
                        'label': 'Pause',
                        'method': 'animate',
                        'args': [
                            [None],
                            {
                                'frame': {'duration': 0, 'redraw': True},
                                'mode': 'immediate',
                                'transition': {'duration': 0},
                            },
                        ],
                    },
                ],
            },
            dict(
                type='dropdown', buttons=direction_buttons, x=0.4, y=1.05,
            ),
        ],
        sliders=[
            {
                'x': 0.08,
                'len': 0.9,
                'currentvalue': {'prefix': 'Time bin (UTC): '},
                'steps': [
                    {
                        'label': pd.Timestamp(frame.name).strftime('%m-%d %H:%M'),
                        'method': 'animate',
                        'args': [
                            [frame.name],
                            {
                                'frame': {'duration': 0, 'redraw': True},
                                'mode': 'immediate',
                                'transition': {'duration': 0},
                            },
                        ],
                    }
                    for frame in frames
                ],
            }
        ],
    )
    return figure

In [ ]:
raw_animation = make_congestion_animation(
    raw_congestion_grid,
    title=f'Route {route_id} Segment Congestion Over Time (last {animation_window})',
    value_label='Congestion index',
)
raw_animation.write_html(
    visuals_dir / 'route_97_segment_congestion_engine_animation.html'
)
#raw_animation.show()

In [ ]:
smoothed_animation = make_congestion_animation(
    smoothed_congestion_grid,
    title=(
        f'Route {route_id} Smoothed Segment Congestion Over Time '
        f'(last {animation_window}, 10-bin rolling mean)'
    ),
    value_label='Smoothed congestion index',
)
smoothed_animation.write_html(
    visuals_dir / 'route_97_smoothed_segment_congestion_engine_animation.html'
)
smoothed_animation.show()

## Segment time series and probe availability

Select a segment ID from the table above. Each series and probe count belongs to exactly one direction and shape, both identified in the title. Timestamps are converted from UTC to Vancouver local time for display. Smoothing is a centered five-bin rolling mean.

In [ ]:
segment_id_of_interest = 8
segment_matches = segments.loc[
    segments['segment_id'].eq(segment_id_of_interest)
]
if segment_matches.empty:
    raise ValueError(f'Segment {segment_id_of_interest} is not present on route {route_id}.')
segment = segment_matches.iloc[0]
segment_direction_id = int(segment['trip_direction_id'])
segment_label = (
    f"Direction {segment_direction_id}, shape {segment['shape_id']}: "
    f"{segment['from_stop_name']} -> {segment['to_stop_name']}"
)

df_single_segment = (
    df_congestion.loc[
        df_congestion['segment_id'].eq(segment_id_of_interest)
        & df_congestion['trip_direction_id'].eq(segment_direction_id)
    ]
    .sort_values('time_bin')
    .copy()
)
df_single_segment['time_bin_local'] = df_single_segment['time_bin'].dt.tz_convert(
    'America/Vancouver'
)
segment_smoothing_window = 5
df_single_segment['congestion_index_smooth'] = (
    df_single_segment['congestion_index']
    .rolling(
        window=segment_smoothing_window,
        min_periods=1,
        center=True,
    )
    .mean()
)

segment_figure = go.Figure()
segment_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index'],
        mode='lines+markers',
        name='Raw',
        line=dict(color='lightgray'),
        opacity=0.6,
    )
)
segment_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index_smooth'],
        mode='lines+markers',
        name=f'{segment_smoothing_window}-bin rolling mean',
        line=dict(color='darkorange'),
    )
)
segment_figure.update_layout(
    title=f'Smoothed Congestion Index for Segment: {segment_label}',
    xaxis_title='Time (Vancouver)',
    yaxis_title='Congestion Index (0 = free flow, 1 = stopped)',
)
segment_figure.show()

In [ ]:
df_probe_counts = (
    vehicle_probes.loc[
        vehicle_probes['segment_id'].eq(segment_id_of_interest)
        & vehicle_probes['trip_direction_id'].eq(segment_direction_id)
    ]
    .groupby('time_bin')['vehicle_id']
    .nunique()
    .rename('probe_count')
    .reset_index()
    .sort_values('time_bin')
)
df_probe_counts['time_bin_local'] = df_probe_counts['time_bin'].dt.tz_convert(
    'America/Vancouver'
)

probe_figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.08,
    subplot_titles=(
        f'Congestion Index for Segment: {segment_label}',
        'Number of Buses (Probes) Used per Time Bin',
    ),
)
probe_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index'],
        mode='lines+markers',
        name='Raw',
        line=dict(color='lightgray'),
        opacity=0.6,
    ),
    row=1,
    col=1,
)
probe_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index_smooth'],
        mode='lines+markers',
        name=f'{segment_smoothing_window}-bin rolling mean',
        line=dict(color='darkorange'),
    ),
    row=1,
    col=1,
)
probe_figure.add_trace(
    go.Bar(
        x=df_probe_counts['time_bin_local'],
        y=df_probe_counts['probe_count'],
        name='Probe count',
        marker=dict(color='steelblue'),
    ),
    row=2,
    col=1,
)
probe_figure.update_yaxes(
    title_text='Congestion Index (0 = free flow, 1 = stopped)',
    row=1,
    col=1,
)
probe_figure.update_yaxes(title_text='# Buses', row=2, col=1)
probe_figure.update_xaxes(title_text='Time (Vancouver)', row=2, col=1)
probe_figure.update_layout(
    title=f'Congestion Index vs. Probe Availability: {segment_label}',
    showlegend=True,
    height=700,
)
probe_figure.show()

## Latest reported delay at Harvey Ave at Gordon Stn

Route `97-KEL` serves two platforms named **Harvey Ave at Gordon Stn** (`140002` and `140003`). Keep both platforms separate. For each trip instance and stop visit, take the row with the latest `feed_timestamp` and use its `arrival_delay` in seconds, converted to minutes for plotting. Do not average earlier predictions or substitute `departure_delay`.

Trip start descriptors distinguish repeated daily trips; stop sequences distinguish repeated visits. Missing start descriptors produce a warning and use Vancouver publication dates for approximate daily grouping. Missing delays in the latest rows remain missing, rather than falling back to older predictions.

This is the latest **reported estimate**, not independently measured ground truth. The last available update can be published after arrival or still be provisional if the data window ends before the visit. The x-axis is the latest update's publication time in Vancouver, matching the congestion plots' timezone. Each platform has raw estimates and a five-visit rolling mean; this smoothing is over visits, not five-minute congestion bins. Positive delays mean late; negative delays mean early.

The saved example output uses the local backtest snapshot covering September 8-22, 2026 (UTC). Rerunning these cells uses whichever `df_trip_updates` is currently loaded. If `trip_direction_id` is selected above, delay estimates are filtered to that same direction.

In [ ]:
#stop_name = 'Hwy 97 at Butt Rd Stn'
#stop_name = 'Hwy 97 at Westside Rd Stn'
#stop_name = 'Queensway Exchange Bay B'
#stop_name = 'Harvey Ave at Richter Stn'
#stop_name = 'Harvey Ave at Gordon Stn'
#stop_name = 'Harvey Ave at Parkinson Stn'
#stop_name = 'Hwy 97 N at Banks Stn'
stop_name = 'Hwy 97 N at Sexsmith Stn'


df_delay_trip_updates = df_trip_updates
if trip_direction_id is not None:
    df_delay_trip_updates = df_trip_updates.loc[
        pd.to_numeric(df_trip_updates['trip_direction_id'], errors='raise')
        .eq(trip_direction_id)
    ]
df_stop_delays = get_latest_stop_delays(
    df_delay_trip_updates,
    df_stops,
    route_id=route_id,
    stop_name=stop_name,
)
df_stop_delays['feed_timestamp_local'] = (
    df_stop_delays['feed_timestamp'].dt.tz_convert('America/Vancouver')
)
delay_smoothing_window = 5
df_stop_delays['delay_minutes_smooth'] = (
    df_stop_delays.groupby('stop_id')['delay_minutes']
    .transform(
        lambda delays: delays.rolling(
            window=delay_smoothing_window,
            min_periods=1,
            center=True,
        ).mean()
    )
    .where(df_stop_delays['delay_minutes'].notna())
)
print(f'Latest trip/stop visits on route {route_id}: {len(df_stop_delays):,}')
print(df_stop_delays.groupby('stop_id').size().rename('visit_count'))
print(f'Visits with missing arrival delay: {df_stop_delays["arrival_delay"].isna().sum():,}')
df_stop_delays[
    ['trip_id', 'trip_start_timestamp', 'stop_id',
     'feed_timestamp_local', 'arrival_delay', 'delay_minutes']
].head(10)

In [ ]:
delay_figure = go.Figure()
delay_colors = ['royalblue', 'darkorange']
for platform_index, (platform_id, platform_delays) in enumerate(
    df_stop_delays.groupby('stop_id', sort=True)
):
    
    direction_id = platform_delays['trip_direction_id'].iloc[0]
    platform_label = f'Stop {platform_id} | Dir {direction_id}'
    color = delay_colors[direction_id % len(delay_colors)]
    delay_figure.add_trace(
        go.Scatter(
            x=platform_delays['feed_timestamp_local'],
            y=platform_delays['delay_minutes'],
            mode='lines+markers',
            name=f'{platform_label}: raw latest estimates',
            legendgroup=platform_id,
            line=dict(color=color, width=1),
            opacity=0.45,
            connectgaps=False,
            customdata=platform_delays[
                ['trip_id', 'trip_start_timestamp', 'arrival_delay']
            ].to_numpy(),
            hovertemplate=(
                f'{platform_label}<br>'
                'Latest update: %{x}<br>'
                'Arrival delay: %{y:.2f} min (%{customdata[2]} s)<br>'
                'Trip: %{customdata[0]}<br>'
                'Trip start: %{customdata[1]}<extra></extra>'
            ),
        )
    )
    delay_figure.add_trace(
        go.Scatter(
            x=platform_delays['feed_timestamp_local'],
            y=platform_delays['delay_minutes_smooth'],
            mode='lines+markers',
            name=f'{platform_label}: {delay_smoothing_window}-visit rolling mean',
            legendgroup=platform_id,
            line=dict(color=color, width=3),
            connectgaps=False,
        )
    )
delay_figure.add_hline(y=0, line_dash='dash', line_color='gray')
delay_figure.update_layout(
    title=f'Latest Reported Arrival Delay: {stop_name} (Route {route_id})',
    xaxis_title='Latest trip-update time (Vancouver)',
    yaxis_title='Arrival delay (minutes; positive = late, negative = early)',
    hovermode='closest',
    legend_title_text='Platform and estimate',
)
delay_plot_path = visuals_dir / 'route_97_late_stop_delays.html'
delay_figure.write_html(delay_plot_path)
print(f'Saved delay time series: {delay_plot_path}')
delay_figure.show()